# Clasificación de Dígitos Manuscritos — TecnoForms
**Curso:** Mantenimiento de Equipos Periféricos (PIAD-427) · **SENATI** · Trabajo Final

Contenido: preprocesamiento y segmentación de imágenes · Regresión logística · K-Means · KNN · SVM · Red neuronal densa · **CNN** · validación cruzada · métricas · resultados visuales · arquitectura · aplicación a un formulario.

## 0. Instalación de librerías (ejecutar una sola vez)

In [ ]:
%pip install -q opencv-python numpy pandas matplotlib scikit-learn tensorflow

## 1. Librerías y parámetros

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)

#PARÁMETROS GENERALES (se pueden reducir para hacer pruebas rápidas)
SEMILLA = 42
N_CLASICOS = 20000            #Muestras de entrenamiento para Reg. Logística, KNN, SVM y K-Means
N_VALIDACION_CRUZADA = 12000  #Muestras usadas en la validación cruzada de la CNN
K_FOLDS = 3                   #Número de particiones de la validación cruzada
EPOCAS_CV = 3                 #Épocas por entrenamiento dentro de la validación cruzada
EPOCAS_FINAL = 12             #Épocas máximas del entrenamiento final de la CNN

np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)
os.makedirs("resultados", exist_ok=True)   #Carpeta donde se guardan gráficos y tablas
print("TensorFlow:", tf.__version__)

## 2. Carga del conjunto de datos (MNIST)

In [ ]:
mnist = fetch_openml("mnist_784", version = 1, as_frame = False)
x = mnist.data.astype("float32")
y = mnist.target.astype("int32")
print(x.shape)
print(y.shape)

In [ ]:
x = x/255 #Normalizamos las imagenes (0 a 1)
x_img = x.reshape(-1,28,28,1) #Lote de 70000 imagenes (28,28) en monocromático para la CNN
print("Forma de x (aplanado):", x.shape)
print("Forma de x_img (imagen):", x_img.shape)

In [ ]:
#División entrenamiento / test (80% - 20%), manteniendo la proporción de cada dígito
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size = 0.2, random_state = SEMILLA, stratify = y)

x_train, x_test = x[idx_train], x[idx_test]              #Aplanadas (784) para los modelos clásicos y la red densa
x_train_img, x_test_img = x_img[idx_train], x_img[idx_test]   #Imágenes (28,28,1) para la CNN
y_train, y_test = y[idx_train], y[idx_test]
print(x_train.shape, x_test.shape)

In [ ]:
plt.figure(figsize = (10,4))
for i in range(10):
  plt.subplot(2,5,i+1)
  plt.imshow(x_train_img[i].reshape(28,28), cmap = "gray")
  plt.title("Etiqueta: " + str(y_train[i]))
  plt.axis("off")
plt.tight_layout()
plt.savefig("resultados/01_muestras_dataset.png", dpi = 150)
plt.show()

## 3. Preprocesamiento y segmentación de imágenes de formularios

In [ ]:
def a_grises(imagen):
  #Conversión a escala de grises (si la imagen viene a color BGR)
  if imagen.ndim == 3:
    return cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
  return imagen

def binarizar(gris):
  #Suavizado gaussiano (reduce ruido) + umbral automático de Otsu
  #THRESH_BINARY_INV: la tinta queda en blanco (255) y el papel en negro (0), igual que MNIST
  suave = cv2.GaussianBlur(gris, (5,5), 0)
  _, binaria = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
  return binaria

def ordenar_cajas(cajas):
  #Ordena las cajas como se lee un formulario: de arriba hacia abajo y de izquierda a derecha
  if len(cajas) == 0:
    return cajas
  cajas = sorted(cajas, key = lambda b: b[1] + b[3]/2)
  lineas, actual = [], [cajas[0]]
  for c in cajas[1:]:
    centro_linea = np.mean([b[1] + b[3]/2 for b in actual])
    alto_linea = np.mean([b[3] for b in actual])
    if abs((c[1] + c[3]/2) - centro_linea) < 0.6*alto_linea:
      actual.append(c)
    else:
      lineas.append(actual)
      actual = [c]
  lineas.append(actual)
  return [b for linea in lineas for b in sorted(linea, key = lambda b: b[0])]

def segmentar_digitos(binaria, alto_min = 15, area_min = 150):
  #Segmentación: detecta cada dígito como un contorno externo y devuelve su caja (x, y, ancho, alto)
  kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
  unida = cv2.dilate(binaria, kernel, iterations = 1) #Une trazos cortados de un mismo dígito
  contornos, _ = cv2.findContours(unida, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
  cajas = [cv2.boundingRect(c) for c in contornos]
  cajas = [b for b in cajas if b[3] >= alto_min and b[2]*b[3] >= area_min] #Elimina manchas y ruido
  return ordenar_cajas(cajas)

def a_formato_mnist(recorte):
  #Lleva el recorte de un dígito al formato MNIST: 28x28, dígito de 20 px de lado mayor y centrado por centro de masa
  h, w = recorte.shape
  escala = 20.0/max(h, w)
  nuevo = cv2.resize(recorte, (max(1, int(round(w*escala))), max(1, int(round(h*escala)))), interpolation = cv2.INTER_AREA)
  lienzo = np.zeros((28,28), np.uint8)
  y0, x0 = (28 - nuevo.shape[0])//2, (28 - nuevo.shape[1])//2
  lienzo[y0:y0 + nuevo.shape[0], x0:x0 + nuevo.shape[1]] = nuevo
  m = cv2.moments(lienzo)
  if m["m00"] > 0:
    cx, cy = m["m10"]/m["m00"], m["m01"]/m["m00"]
    lienzo = cv2.warpAffine(lienzo, np.float32([[1,0,14 - cx],[0,1,14 - cy]]), (28,28))
  return lienzo.astype("float32")/255 #Normalización 0 a 1

def procesar_formulario(imagen):
  #Pipeline completo: imagen -> grises -> binaria -> segmentación -> dígitos 28x28 normalizados
  gris = a_grises(imagen)
  binaria = binarizar(gris)
  cajas = segmentar_digitos(binaria)
  digitos = [a_formato_mnist(binaria[y0:y0 + h, x0:x0 + w]) for (x0, y0, w, h) in cajas]
  digitos = np.array(digitos).reshape(-1,28,28,1) if len(digitos) > 0 else np.zeros((0,28,28,1), "float32")
  return gris, binaria, cajas, digitos

In [ ]:
def crear_formulario(imagenes, columnas = 6, tam = 80, margen = 40):
  #Genera un formulario ficticio (papel con ruido e iluminación irregular) con los dígitos escritos en casillas
  filas = int(np.ceil(len(imagenes)/columnas))
  alto, ancho = filas*(tam + margen) + margen, columnas*(tam + margen) + margen
  papel = np.full((alto, ancho), 225, np.float32)
  for i, img in enumerate(imagenes):
    grande = cv2.resize((img.reshape(28,28)*255).astype("uint8"), (tam, tam), interpolation = cv2.INTER_CUBIC)
    f, c = divmod(i, columnas)
    y0 = margen + f*(tam + margen) + np.random.randint(-6, 7)
    x0 = margen + c*(tam + margen) + np.random.randint(-6, 7)
    papel[y0:y0 + tam, x0:x0 + tam] *= (1 - 0.9*grande/255.0)   #La tinta oscurece el papel
  gradiente = np.linspace(-20, 20, ancho)[None, :]             #Iluminación irregular
  papel = papel + gradiente + np.random.normal(0, 7, papel.shape) #Ruido
  papel = np.clip(papel, 0, 255).astype("uint8")
  return cv2.cvtColor(papel, cv2.COLOR_GRAY2BGR)

#Formulario ficticio con 12 dígitos del conjunto de test
formulario = crear_formulario(x_test_img[:12])
etiquetas_formulario = y_test[:12]

gris, binaria, cajas, digitos = procesar_formulario(formulario)
print("Dígitos detectados:", len(cajas), "| Dígitos reales:", len(etiquetas_formulario))

con_cajas = formulario.copy()
for (x0, y0, w, h) in cajas:
  cv2.rectangle(con_cajas, (x0, y0), (x0 + w, y0 + h), (0, 180, 0), 2)

plt.figure(figsize = (16,8))
plt.subplot(2,2,1); plt.imshow(cv2.cvtColor(formulario, cv2.COLOR_BGR2RGB)); plt.title("1. Imagen original"); plt.axis("off")
plt.subplot(2,2,2); plt.imshow(gris, cmap = "gray"); plt.title("2. Escala de grises"); plt.axis("off")
plt.subplot(2,2,3); plt.imshow(binaria, cmap = "gray"); plt.title("3. Binarización (Otsu)"); plt.axis("off")
plt.subplot(2,2,4); plt.imshow(cv2.cvtColor(con_cajas, cv2.COLOR_BGR2RGB)); plt.title("4. Segmentación de dígitos"); plt.axis("off")
plt.tight_layout()
plt.savefig("resultados/02_preprocesamiento_formulario.png", dpi = 150)
plt.show()

plt.figure(figsize = (12,2))
for i in range(len(digitos)):
  plt.subplot(1, len(digitos), i+1)
  plt.imshow(digitos[i].reshape(28,28), cmap = "gray")
  plt.axis("off")
plt.suptitle("5. Dígitos normalizados a 28x28 (entrada del modelo)")
plt.savefig("resultados/03_digitos_segmentados.png", dpi = 150)
plt.show()

## 4. Modelos de comparación (Regresión, K-Means, KNN, SVM y red densa)

In [ ]:
#Tabla donde se guardan las métricas de todos los modelos
resultados = []
def registrar(nombre, y_real, y_pred):
  resultados.append({
      "Modelo": nombre,
      "Exactitud (accuracy)": accuracy_score(y_real, y_pred),
      "Precisión (macro)": precision_score(y_real, y_pred, average = "macro", zero_division = 0),
      "Recall (macro)": recall_score(y_real, y_pred, average = "macro", zero_division = 0),
      "F1 (macro)": f1_score(y_real, y_pred, average = "macro", zero_division = 0)})
  print(nombre, "-> accuracy:", round(accuracy_score(y_real, y_pred), 4))

#Reducción de dimensionalidad con PCA (784 -> 50) para acelerar los modelos clásicos
rng = np.random.default_rng(SEMILLA)
n = min(N_CLASICOS, len(x_train))
idx = rng.choice(len(x_train), n, replace = False)
y_train_sub = y_train[idx]

pca = PCA(n_components = 50, random_state = SEMILLA)
x_train_pca = pca.fit_transform(x_train[idx])
x_test_pca = pca.transform(x_test)
print("Varianza explicada por 50 componentes:", round(pca.explained_variance_ratio_.sum(), 4))

In [ ]:
#REGRESIÓN LOGÍSTICA (regresión softmax multiclase)
reg_log = LogisticRegression(max_iter = 300)
reg_log.fit(x_train_pca, y_train_sub)
y_pred_reg = reg_log.predict(x_test_pca)
registrar("Regresión logística", y_test, y_pred_reg)

In [ ]:
#K-NEAREST NEIGHBORS (KNN)
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(x_train_pca, y_train_sub)
y_pred_knn = knn.predict(x_test_pca)
registrar("KNN (k=3)", y_test, y_pred_knn)

In [ ]:
#SUPPORT VECTOR MACHINE (SVM) con kernel RBF
svm = SVC(kernel = "rbf", C = 5, gamma = "scale")
svm.fit(x_train_pca, y_train_sub)
y_pred_svm = svm.predict(x_test_pca)
registrar("SVM (RBF)", y_test, y_pred_svm)

In [ ]:
#K-MEANS (aprendizaje no supervisado): agrupa las imágenes en 10 clusters sin usar las etiquetas
kmeans = KMeans(n_clusters = 10, n_init = 10, random_state = SEMILLA)
kmeans.fit(x_train_pca)

#Para clasificar, cada cluster toma la etiqueta mayoritaria de las imágenes de entrenamiento que contiene
mapa_cluster = {c: np.bincount(y_train_sub[kmeans.labels_ == c], minlength = 10).argmax() for c in range(10)}
y_pred_km = np.array([mapa_cluster[c] for c in kmeans.predict(x_test_pca)])
registrar("K-Means (10 clusters)", y_test, y_pred_km)

#Centroides de cada cluster (el "dígito promedio" que aprendió K-Means)
centroides = pca.inverse_transform(kmeans.cluster_centers_)
plt.figure(figsize = (10,4))
for c in range(10):
  plt.subplot(2,5,c+1)
  plt.imshow(centroides[c].reshape(28,28), cmap = "gray")
  plt.title("Cluster " + str(c) + " -> " + str(mapa_cluster[c]))
  plt.axis("off")
plt.tight_layout()
plt.savefig("resultados/04_centroides_kmeans.png", dpi = 150)
plt.show()

In [ ]:
#RED NEURONAL DENSA (MLP) - entrada aplanada de 784 pixeles
mlp = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation = "relu"),
    tf.keras.layers.Dense(64, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])
mlp.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])
mlp.fit(x_train, y_train, epochs = 10, batch_size = 128, verbose = 2)
y_pred_mlp = mlp.predict(x_test, verbose = 0).argmax(1)
registrar("Red neuronal densa (MLP)", y_test, y_pred_mlp)

## 5. CNN: validación cruzada y ajuste de hiperparámetros

In [ ]:
def crear_cnn(filtros = 16, unidades = 64, dropout = 0.3, lr = 0.001):
  #RNA CONVOLUCIONAL: 3 bloques (Conv2D + BatchNormalization + MaxPooling2D) y capas densas para la clasificación
  modelo = tf.keras.models.Sequential([
      tf.keras.layers.Input(shape=(28,28,1)),

      tf.keras.layers.Conv2D(filtros, 3, padding = "same", activation = "relu"),
      tf.keras.layers.BatchNormalization(momentum = 0.9),
      tf.keras.layers.MaxPooling2D(),

      tf.keras.layers.Conv2D(filtros*2, 3, padding = "same", activation = "relu"),
      tf.keras.layers.BatchNormalization(momentum = 0.9),
      tf.keras.layers.MaxPooling2D(),

      tf.keras.layers.Conv2D(filtros*4, 3, padding = "same", activation = "relu"),
      tf.keras.layers.BatchNormalization(momentum = 0.9),
      tf.keras.layers.MaxPooling2D(),

      tf.keras.layers.Flatten(),
      tf.keras.layers.Dropout(dropout),
      tf.keras.layers.Dense(unidades, activation = "relu"),
      tf.keras.layers.Dense(10, activation = "softmax")
  ])
  modelo.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
                 loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])
  return modelo

def validacion_cruzada(parametros, x_cv, y_cv, k = K_FOLDS, epocas = EPOCAS_CV):
  #Validación cruzada estratificada de k particiones: devuelve la exactitud media y su desviación
  skf = StratifiedKFold(n_splits = k, shuffle = True, random_state = SEMILLA)
  exactitudes = []
  for idx_ent, idx_val in skf.split(x_cv, y_cv):
    modelo = crear_cnn(**parametros)
    modelo.fit(x_cv[idx_ent], y_cv[idx_ent], epochs = epocas, batch_size = 64, verbose = 0)
    exactitudes.append(modelo.evaluate(x_cv[idx_val], y_cv[idx_val], verbose = 0)[1])
    tf.keras.backend.clear_session()
  return np.mean(exactitudes), np.std(exactitudes)

In [ ]:
#BÚSQUEDA DE HIPERPARÁMETROS (grid search) con validación cruzada
rejilla = {
    "filtros": [16, 32],
    "unidades": [64, 128],
    "lr": [0.001, 0.003]
}

n_cv = min(N_VALIDACION_CRUZADA, len(x_train_img))
idx_cv = np.random.default_rng(SEMILLA).choice(len(x_train_img), n_cv, replace = False)
x_cv, y_cv = x_train_img[idx_cv], y_train[idx_cv]

filas = []
for valores in itertools.product(*rejilla.values()):
  parametros = dict(zip(rejilla.keys(), valores))
  media, desviacion = validacion_cruzada(parametros, x_cv, y_cv)
  filas.append({**parametros, "exactitud_media": media, "desviacion": desviacion})
  print(parametros, "-> exactitud:", round(media, 4), "+/-", round(desviacion, 4))

tabla_cv = pd.DataFrame(filas).sort_values("exactitud_media", ascending = False).reset_index(drop = True)
tabla_cv.to_csv("resultados/validacion_cruzada.csv", index = False)
tabla_cv

In [ ]:
mejores = tabla_cv.loc[0, ["filtros", "unidades", "lr"]].to_dict()
mejores = {"filtros": int(mejores["filtros"]), "unidades": int(mejores["unidades"]), "lr": float(mejores["lr"])}
print("Mejores hiperparámetros:", mejores)

## 6. Entrenamiento final de la CNN

In [ ]:
#Conjunto de validación (10% del entrenamiento) para controlar el sobreajuste
x_ent, x_val, y_ent, y_val = train_test_split(x_train_img, y_train, test_size = 0.1, random_state = SEMILLA, stratify = y_train)

#Aumento de datos (rotación, desplazamiento y zoom leves) para que el modelo tolere distintas formas de escribir
aumento = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.08, fill_mode = "constant", fill_value = 0.0),
    tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode = "constant", fill_value = 0.0),
    tf.keras.layers.RandomZoom(0.1, fill_mode = "constant", fill_value = 0.0)
])

ds_ent = (tf.data.Dataset.from_tensor_slices((x_ent, y_ent))
          .shuffle(10000, seed = SEMILLA)
          .batch(128)
          .map(lambda a, b: (aumento(a, training = True), b), num_parallel_calls = tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))
ds_val = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(128)

modelo = crear_cnn(**mejores)
parada = tf.keras.callbacks.EarlyStopping(monitor = "val_accuracy", patience = 3, restore_best_weights = True)
historial = modelo.fit(ds_ent, epochs = EPOCAS_FINAL, validation_data = ds_val, callbacks = [parada], shuffle = False, verbose = 2)

In [ ]:
plt.figure(figsize = (12,4))
plt.subplot(1,2,1)
plt.plot(historial.history["accuracy"], label = "Entrenamiento")
plt.plot(historial.history["val_accuracy"], label = "Validación")
plt.title("Exactitud por época"); plt.xlabel("Época"); plt.ylabel("Exactitud"); plt.legend()
plt.subplot(1,2,2)
plt.plot(historial.history["loss"], label = "Entrenamiento")
plt.plot(historial.history["val_loss"], label = "Validación")
plt.title("Pérdida por época"); plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.legend()
plt.tight_layout()
plt.savefig("resultados/05_curvas_entrenamiento.png", dpi = 150)
plt.show()

## 7. Arquitectura de la red neuronal

In [ ]:
modelo.summary()

In [ ]:
#Tabla de arquitectura: capas, nodos/filtros, función de activación y parámetros
capas = []
for capa in modelo.layers:
  tipo = capa.__class__.__name__
  nodos = getattr(capa, "filters", None) or getattr(capa, "units", None) or "-"
  activacion = getattr(getattr(capa, "activation", None), "__name__", "-")
  try:
    salida = str(capa.output.shape[1:])
  except Exception:
    salida = "-"
  capas.append({"Capa": capa.name, "Tipo": tipo, "Nodos / Filtros": nodos, "Activación": activacion,
                "Forma de salida": salida, "Parámetros": capa.count_params()})
tabla_arquitectura = pd.DataFrame(capas)
tabla_arquitectura.to_csv("resultados/arquitectura_cnn.csv", index = False)
display(tabla_arquitectura)

print("Total de capas:", len(modelo.layers))
print("Total de parámetros:", modelo.count_params())
print("Optimizador: Adam (learning_rate =", mejores["lr"], ")")
print("Función de pérdida: sparse_categorical_crossentropy")
print("Regularización: Dropout, BatchNormalization y aumento de datos")

## 8. Evaluación en el conjunto de test

In [ ]:
probs = modelo.predict(x_test_img, verbose = 0)
y_pred_cnn = probs.argmax(1)
registrar("CNN (red convolucional)", y_test, y_pred_cnn)

print()
print(classification_report(y_test, y_pred_cnn, digits = 4)) #Precisión, recall y F1 por dígito

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_cnn, ax = ax, cmap = "Blues", colorbar = False)
ax.set_title("Matriz de confusión - CNN (conjunto de test)")
plt.savefig("resultados/06_matriz_confusion_cnn.png", dpi = 150, bbox_inches = "tight")
plt.show()

## 9. Comparación de modelos

In [ ]:
tabla_modelos = pd.DataFrame(resultados).sort_values("Exactitud (accuracy)", ascending = False).reset_index(drop = True)
tabla_modelos.to_csv("resultados/comparacion_modelos.csv", index = False)
display(tabla_modelos.round(4))

plt.figure(figsize = (10,5))
plt.barh(tabla_modelos["Modelo"][::-1], tabla_modelos["Exactitud (accuracy)"][::-1], color = "steelblue")
plt.xlim(0, 1)
for i, v in enumerate(tabla_modelos["Exactitud (accuracy)"][::-1]):
  plt.text(v + 0.005, i, f"{v:.4f}", va = "center")
plt.title("Exactitud en el conjunto de test por modelo")
plt.xlabel("Exactitud")
plt.tight_layout()
plt.savefig("resultados/07_comparacion_modelos.png", dpi = 150)
plt.show()

In [ ]:
#Matrices de confusión de los modelos alternativos
modelos_pred = {"Regresión logística": y_pred_reg, "KNN (k=3)": y_pred_knn, "SVM (RBF)": y_pred_svm,
                "K-Means": y_pred_km, "Red densa (MLP)": y_pred_mlp, "CNN": y_pred_cnn}
fig, ejes = plt.subplots(2, 3, figsize = (18,12))
for ax, (nombre, pred) in zip(ejes.ravel(), modelos_pred.items()):
  ConfusionMatrixDisplay.from_predictions(y_test, pred, ax = ax, cmap = "Blues", colorbar = False)
  ax.set_title(nombre)
plt.tight_layout()
plt.savefig("resultados/08_matrices_confusion_todos.png", dpi = 120)
plt.show()

## 10. Resultados visuales: predicciones vs etiquetas reales

In [ ]:
plt.figure(figsize = (14,7))
for i in range(24):
  plt.subplot(3,8,i+1)
  plt.imshow(x_test_img[i].reshape(28,28), cmap = "gray")
  correcto = y_test[i] == y_pred_cnn[i]
  plt.title("Real: " + str(y_test[i]) + "\nPred: " + str(y_pred_cnn[i]), color = "green" if correcto else "red", fontsize = 9)
  plt.axis("off")
plt.suptitle("Predicciones de la CNN (verde = acierto, rojo = error)")
plt.tight_layout()
plt.savefig("resultados/09_predicciones_vs_reales.png", dpi = 150)
plt.show()

In [ ]:
#Errores del modelo
errores = np.where(y_pred_cnn != y_test)[0]
print("Imágenes mal clasificadas:", len(errores), "de", len(y_test))

plt.figure(figsize = (14,4))
for j, i in enumerate(errores[:16]):
  plt.subplot(2,8,j+1)
  plt.imshow(x_test_img[i].reshape(28,28), cmap = "gray")
  plt.title("Real: " + str(y_test[i]) + "\nPred: " + str(y_pred_cnn[i]) + " (" + str(int(probs[i].max()*100)) + "%)", color = "red", fontsize = 8)
  plt.axis("off")
plt.suptitle("Ejemplos de errores de la CNN")
plt.tight_layout()
plt.savefig("resultados/10_errores_cnn.png", dpi = 150)
plt.show()

## 11. Aplicación completa: clasificación automática de un formulario

In [ ]:
#Imagen del formulario -> preprocesamiento -> segmentación -> CNN
#Para usar un formulario real, reemplazar por:  formulario = cv2.imread("mi_formulario.jpg")
gris, binaria, cajas, digitos = procesar_formulario(formulario)
probs_form = modelo.predict(digitos, verbose = 0)
pred_form = probs_form.argmax(1)

resultado = formulario.copy()
for (x0, y0, w, h), p, conf in zip(cajas, pred_form, probs_form.max(1)):
  cv2.rectangle(resultado, (x0, y0), (x0 + w, y0 + h), (0, 180, 0), 2)
  cv2.putText(resultado, str(p), (x0, y0 - 8), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

plt.figure(figsize = (14,6))
plt.imshow(cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB))
plt.title("Formulario procesado: dígitos detectados y predichos")
plt.axis("off")
plt.savefig("resultados/11_formulario_clasificado.png", dpi = 150, bbox_inches = "tight")
plt.show()

print("Predichos:", pred_form)
if len(pred_form) == len(etiquetas_formulario):
  print("Reales:   ", etiquetas_formulario)
  print("Exactitud en el formulario:", accuracy_score(etiquetas_formulario, pred_form))

## 12. Guardar el modelo y exportarlo para la página web

In [ ]:
#Exporta la CNN entrenada a un archivo JavaScript (modelo.js) que lee la página web
import json
def exportar_modelo_web(modelo, ruta = "modelo.js"):
  capas = []
  for capa in modelo.layers:
    tipo = capa.__class__.__name__
    pesos = capa.get_weights()
    info = {"formas": [list(p.shape) for p in pesos],
            "pesos": [np.round(p, 5).flatten().tolist() for p in pesos]}
    if tipo == "Conv2D":
      info.update(tipo = "conv", filtros = capa.filters, activacion = capa.activation.__name__)
    elif tipo == "BatchNormalization":
      info.update(tipo = "bn", epsilon = float(capa.epsilon))
    elif tipo == "MaxPooling2D":
      info.update(tipo = "pool")
    elif tipo == "Flatten":
      info.update(tipo = "flatten")
    elif tipo == "Dense":
      info.update(tipo = "dense", unidades = capa.units, activacion = capa.activation.__name__)
    else:
      continue #Dropout no se usa al predecir
    capas.append(info)
  with open(ruta, "w") as f:
    f.write("const MODELO = " + json.dumps(capas, separators = (",", ":")) + ";")
  print("Modelo exportado:", ruta, "(", round(os.path.getsize(ruta)/1024), "KB )")

In [ ]:
modelo.save("modelo_digitos.keras")
print("Modelo guardado: modelo_digitos.keras")

exportar_modelo_web(modelo, "modelo.js") #Este archivo se coloca junto a index.html en la página web

try:
  from google.colab import files
  files.download("modelo_digitos.keras")
  files.download("modelo.js")
except ImportError:
  pass